In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt


In [36]:
class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  #16,28,28
            nn.Dropout(p=0.2)
        )
        self.layer2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.layer3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.layer4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  #64,28,28

        self.layer_1 = nn.Conv2d(1, 16, kernel_size=1,padding=0) 
        self.layer_2 = nn.Conv2d(16, 32, kernel_size=1,padding=0)
        self.layer_3 = nn.Conv2d(32, 64, kernel_size=1,padding=0)
        self.layer_4 = nn.Conv2d(64, 64, kernel_size=1,padding=0) 
        
        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.relu = nn.ReLU()
        self.max = nn.MaxPool2d(2,2) 
        self.fc1 = nn.Linear(64*7*7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
            identity = x
            x = self.relu(self.layer1(x) + self.layer_1(identity))
            identity = x
            x = self.relu(self.layer2(x) + self.layer_2(identity))
            identity = x
            x = self.relu(self.layer3(x) + self.layer_3(identity))
            identity = x
            x = self.relu(self.layer4(x) + self.layer_4(identity))
         
            x = self.layer5(x)  #64,14,14
            x = self.max(x)  #64,7,7
            x = self.relu(x)
            x = x.flatten(1)
            x = self.fc1(x)
            x = self.fc2(x)
            return x
           
            

In [37]:
train_data = datasets.MNIST(
    root="../data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [transforms.ToTensor()]
    )
)
test_data = datasets.MNIST(
    root="../data",
    train=False,
    download=True,
    transform=transforms.Compose(
        [transforms.ToTensor()]
    )
)
train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=64, 
    shuffle=True
)
## 加载 MNIST 测试数据集
test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=64, 
    shuffle=True
)

image, label = next(iter(train_loader))
print(image.shape)
print(label.shape)
## 检测设备并初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)
print(model)

torch.Size([64, 1, 28, 28])
torch.Size([64])
ResNet(
  (layer1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Dropout(p=0.2, inplace=False)
  )
  (layer2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer_1): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1))
  (layer_2): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
  (layer_3): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
  (layer_4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
  (layer5): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (relu): ReLU()
  (max): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(i

In [38]:
for epoch in range(10):
  model.train()
  total_loss = 0.0
  total_count = 0
  for input, target in train_loader:
    input = input.to(device)
    target = target.to(device)

    optimizer.zero_grad()

    output = model(input)

    loss = loss_fn(output, target)

    loss.backward()

    optimizer.step()

    total_loss += loss.item() * input.size(0)
    total_count += input.size(0)
  avg_loss = total_loss / total_count
  
  if (epoch+1) %10 == 0:
    print("Epoch", epoch+1)
    print("avg loss", avg_loss)


Epoch 10
avg loss 0.0157864985847089
